# The reproduction gate, in miniature

This notebook regenerates **every** figure of the demo from the f3dasm record
`data/` and nothing else. No pickles, no `.npy` files, no numbers typed by hand.

If a result cannot be redrawn from here, it is not a result — it is an anecdote.


In [1]:
import matplotlib
matplotlib.use("Agg")   # scripted execution: save figures, never show them

import matplotlib.pyplot as plt
import numpy as np
from f3dasm import ExperimentData

from make_data import X_HIGH, X_LOW, true_mean, true_sd

data = ExperimentData.from_file("data")
input_df, output_df = data.to_pandas()
x = input_df["x"].to_numpy(float)
y = output_df["y"].to_numpy(float)
grid = np.linspace(X_LOW, X_HIGH, 400)

print(f"record data/: {len(data)} rows")
print("output columns:", list(output_df.columns))

record data/: 60 rows
output columns: ['y', 'y_pred_baseline', 'sd_baseline', '_source_baseline', 'y_pred_model', 'sd_model', '_source_modeler', 'y_pred_selected', 'sd_selected', '_source_selected']


## Baseline (written by `baseline.py`)

Plotted from the stored columns `y_pred_baseline` and `sd_baseline` — the
notebook does not re-fit anything here.

In [2]:
order = np.argsort(x)
mu_b = output_df["y_pred_baseline"].to_numpy(float)[order]
sd_b = float(output_df["sd_baseline"].iloc[0])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, s=18, color="#333333", zorder=3, label="training data")
ax.plot(x[order], mu_b, color="#1f77b4", lw=2, label="baseline mean (deg 2)")
ax.fill_between(x[order], mu_b - 2 * sd_b, mu_b + 2 * sd_b, color="#1f77b4",
                alpha=0.20, label=f"baseline $\\pm2$ sd (constant, sd = {sd_b:.1f})")
ax.plot(grid, true_mean(grid) + 2 * true_sd(grid), "k--", lw=1.4,
        label="true $\\pm2$ sd (sd = 0.5 x)")
ax.plot(grid, true_mean(grid) - 2 * true_sd(grid), "k--", lw=1.4)
ax.set_xlabel("speed x [m/s]"); ax.set_ylabel("stopping distance y [m]")
ax.set_title("Baseline: right mean, wrong noise")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout(); fig.savefig("figures/baseline.png", dpi=150)
print("redrew figures/baseline.png from the record")

redrew figures/baseline.png from the record


---

## Agent sections below

Whoever changes the model or selects its hyperparameters appends cells here,
**between this marker and the final cell**. Those cells may only read columns
that are already in the record — `y_pred_hblr`, `sd_hblr`, `y_pred_selected`,
`sd_selected`, … — plus `data/` itself. They must not read `data_test/`, and
they must not load anything from outside the record.

<!-- AGENT CELLS BELOW THIS LINE -->

### The model that lets the noise fan (written by `blocks/heteroscedastic.py`)

The baseline fitted the mean by least squares and then called the residual
spread "the noise" — one number. Least squares *assumes* constant noise, so
that pipeline could not have found anything else, however hard it looked.

The replacement fits mean and noise **together**, by maximising the Gaussian
log-likelihood, with

$$\mu(x)=\sum_{j=0}^{d_\text{mean}} a_j x^j,
\qquad \log \mathrm{sd}(x)=\sum_{k=0}^{d_\text{noise}} b_k (\log x)^k .$$

The log-link keeps `sd` positive, and expanding it in $\log x$ makes the family
nest both the old answer and the true one: $d_\text{noise}=0$ is a constant
band (the baseline), $d_\text{noise}=1$ is a power law $\mathrm{sd}(x)=e^{b_0}x^{b_1}$,
and the truth of this problem, $\mathrm{sd}[y|x]=0.5x$, is the member with
$e^{b_0}=0.5$, $b_1=1$. The fit was not told that.

`sd_model` is now a **column**, one value per row, not a repeated scalar — so
the figure below is drawn straight from the record, with no re-fitting.

In [3]:
mu_m = output_df["y_pred_model"].to_numpy(float)[order]
sd_m = output_df["sd_model"].to_numpy(float)[order]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, s=18, color="#333333", zorder=3, label="training data")
ax.plot(x[order], mu_m, color="#d62728", lw=2, label="MLE mean")
ax.fill_between(x[order], mu_m - 2 * sd_m, mu_m + 2 * sd_m, color="#d62728",
                alpha=0.20, label="MLE $\\pm2$ sd (fanning)")
ax.plot(grid, true_mean(grid) + 2 * true_sd(grid), "k--", lw=1.4,
        label="true $\\pm2$ sd (sd = 0.5 x)")
ax.plot(grid, true_mean(grid) - 2 * true_sd(grid), "k--", lw=1.4)
ax.plot(x[order], mu_b + 2 * sd_b, color="#1f77b4", lw=1.2, ls=":",
        label="baseline $\\pm2$ sd (constant)")
ax.plot(x[order], mu_b - 2 * sd_b, color="#1f77b4", lw=1.2, ls=":")
ax.set_xlabel("speed x [m/s]"); ax.set_ylabel("stopping distance y [m]")
ax.set_title("Heteroscedastic MLE: the band fans with the truth")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout(); fig.savefig("figures/model.png", dpi=150)
print("redrew figures/model.png from the record")

redrew figures/model.png from the record


In [4]:
# Both models are stored as (mean, sd) per row, so the record can score them
# against each other without knowing how either was fitted.
def mean_log_density(mu, sd):
    return float(np.mean(-np.log(sd) - 0.5 * ((y - mu) / sd) ** 2
                         - 0.5 * np.log(2 * np.pi)))

mu_b_all = output_df["y_pred_baseline"].to_numpy(float)
sd_b_all = output_df["sd_baseline"].to_numpy(float)
mu_m_all = output_df["y_pred_model"].to_numpy(float)
sd_m_all = output_df["sd_model"].to_numpy(float)

lpd_b = mean_log_density(mu_b_all, sd_b_all)
lpd_m = mean_log_density(mu_m_all, sd_m_all)

print("mean log predictive density on data/ (higher is better)")
print(f"  baseline (constant sd) = {lpd_b:.4f}")
print(f"  heteroscedastic MLE    = {lpd_m:.4f}   "
      f"(gain {lpd_m - lpd_b:+.4f} nats/point)")

# Enough digits to see these are two different numbers: they agree to 4 dp by
# coincidence.  The MLE mean is very slightly worse in RMSE on purpose -- it
# weights each point by 1/sd(x)^2, so it stops chasing the noisy fast cars.
print(f"\nRMSE of the mean:  baseline "
      f"{np.sqrt(np.mean((y - mu_b_all) ** 2)):.6f}   "
      f"model {np.sqrt(np.mean((y - mu_m_all) ** 2)):.6f}")
print(f"  max |mu_baseline - mu_model| = "
      f"{np.max(np.abs(mu_b_all - mu_m_all)):.4f} m over the whole record")
print("  -- the means are all but identical; the entire gain is the noise model.")

# What the noise model claims, versus the truth, at the ends of the range.
print("\nsd(x) at the extremes of the record:")
for i in (order[0], order[-1]):
    print(f"  x = {x[i]:5.2f}:  baseline {sd_b_all[i]:7.3f}   "
          f"model {sd_m_all[i]:7.3f}   truth 0.5x {float(true_sd(x[i])):7.3f}")

mean log predictive density on data/ (higher is better)
  baseline (constant sd) = -4.7104
  heteroscedastic MLE    = -4.3096   (gain +0.4008 nats/point)

RMSE of the mean:  baseline 26.865266   model 26.865286
  max |mu_baseline - mu_model| = 0.0819 m over the whole record
  -- the means are all but identical; the entire gain is the noise model.

sd(x) at the extremes of the record:
  x =  3.00:  baseline  27.563   model   1.361   truth 0.5x   1.500
  x = 81.75:  baseline  27.563   model  44.794   truth 0.5x  40.875


### Choosing $(d_\text{mean}, d_\text{noise})$ (written by `blocks/selection.py`)

The model above fixed the noise, but *I* picked its two degrees — the same sin
as the baseline, committed with a better model. The training log-likelihood
cannot arbitrate: it is the very quantity the fit maximises, so it improves
with every parameter added.

So the 12 candidates were scored by **5-fold × 5-repeat cross-validation on
`data/` alone** — each fit sees 48 points, each score uses the 12 it never saw
— and the criterion is the held-out **log predictive density**, a *proper*
scoring rule that grades the whole predictive distribution rather than just its
centre. The study is its own record, `study_selection/`, so the table below is
read back from disk rather than recomputed here.

Two things had to be right before the table meant anything:

- **Conditioning.** A raw design $[1, x, \dots, x^d]$ on $x\in[3,83]$ has
  condition number $10^2$ at $d=1$ but $10^8$ at $d=4$. BFGS stalled and
  reported failure on 12 of 25 folds for the eventual winner and 25 of 25 for
  others — biased *against* the flexible candidates, which would have rigged
  the comparison in favour of the answer I had already guessed. The optimiser
  now runs on a QR-orthonormalised design; `n_fit_failures` is 0 everywhere
  below, and the fitted likelihood is monotone in the number of parameters
  within each nested family, as it must be.
- **Not over-reading the table.** The top cells sit within fold noise of one
  another, so the plain argmax is not a fact. The **one-standard-error rule**
  — keep every candidate within 1 se of the best, then take the fewest
  parameters — is the tiebreak.

In [5]:
study = ExperimentData.from_file("study_selection")
study_in, study_out = study.to_pandas()
study_df = study_in.join(study_out).sort_values("cv_log_pred_density",
                                                ascending=False)

print("study_selection/ -- the 12 candidates, best first")
print(study_df.to_string(index=False))

best, runner = study_df.iloc[0], study_df.iloc[1]
gap = float(best["cv_log_pred_density"] - runner["cv_log_pred_density"])
pooled_se = float(np.hypot(best["cv_lpd_se"], runner["cv_lpd_se"]))
print(f"\ntop two differ by {gap:.4f} nats/point, pooled se {pooled_se:.4f}"
      f"  ->  {'separated' if gap > 2 * pooled_se else 'NOT separated'}")

threshold = float(best["cv_log_pred_density"] - best["cv_lpd_se"])
within = study_df[study_df["cv_log_pred_density"] >= threshold]
pick = within.sort_values(["n_params", "cv_log_pred_density"],
                          ascending=[True, False]).iloc[0]
print(f"one-se rule: {len(within)} candidate(s) within {threshold:.4f}; "
      f"fewest parameters -> d_mean = {int(pick['d_mean'])}, "
      f"d_noise = {int(pick['d_noise'])}")
print(f"total fits behind this table: "
      f"{len(study_df)} candidates x 25 folds = {len(study_df) * 25}")
print(f"optimiser failures across all of them: "
      f"{int(study_df['n_fit_failures'].sum())}")

study_selection/ -- the 12 candidates, best first
 d_mean  d_noise  cv_log_pred_density  cv_lpd_se   cv_rmse  n_params  n_fit_failures _source_selection
      2        1            -4.429031   0.015330 27.828325         5               0         selection
      3        1            -4.476395   0.031548 28.152560         6               0         selection
      2        2            -4.507796   0.043466 27.867381         6               0         selection
      3        2            -4.628038   0.128153 28.217646         7               0         selection
      4        1            -4.669103   0.054554 28.870444         7               0         selection
      2        0            -4.849147   0.019392 28.113636         4               0         selection
      3        0            -4.876606   0.024387 28.623609         5               0         selection
      4        2            -4.877913   0.204006 28.930560         8               0         selection
      4        0       

In [6]:
from blocks.selection import selection_figure

selection_figure(study_df, int(pick["d_mean"]), int(pick["d_noise"]))

# The left panel is the criterion; the right panel is what would have happened
# had we scored the mean only.  RMSE spends its whole dynamic range separating
# d_mean = 1 from the rest, leaving rows 2-4 a flat block in which d_noise --
# the entire question at issue -- is invisible.
row = study_df[study_df["d_mean"] == int(pick["d_mean"])]
print(f"\nalong the d_mean = {int(pick['d_mean'])} row:")
print(f"  cv_rmse spans {row['cv_rmse'].min():.3f} to "
      f"{row['cv_rmse'].max():.3f} m  "
      f"({100 * (row['cv_rmse'].max() / row['cv_rmse'].min() - 1):.1f}%)")
print(f"  cv lpd  spans {row['cv_log_pred_density'].min():.3f} to "
      f"{row['cv_log_pred_density'].max():.3f} nats")

  wrote figures/selection.png

along the d_mean = 2 row:
  cv_rmse spans 27.828 to 28.114 m  (1.0%)
  cv lpd  spans -4.849 to -4.429 nats


In [7]:
# The selected model's own columns, refitted on all 60 points and stored.
mu_s = output_df["y_pred_selected"].to_numpy(float)
sd_s = output_df["sd_selected"].to_numpy(float)

print("stamp:", output_df["_source_selected"].iloc[0])
print(f"mean log predictive density on data/: "
      f"baseline {mean_log_density(mu_b_all, sd_b_all):.4f}   "
      f"selected {mean_log_density(mu_s, sd_s):.4f}")
print(f"max |mu_selected - mu_model| = "
      f"{np.max(np.abs(mu_s - mu_m_all)):.2e} m")
print("  (the selection landed on the same degrees the modeller guessed --")
print("   the difference is that now there is a table saying why.)")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, s=18, color="#333333", zorder=3, label="training data")
ax.plot(x[order], mu_s[order], color="#d62728", lw=2, label="selected mean")
ax.fill_between(x[order], (mu_s - 2 * sd_s)[order], (mu_s + 2 * sd_s)[order],
                color="#d62728", alpha=0.20, label="selected $\\pm2$ sd")
ax.plot(grid, true_mean(grid) + 2 * true_sd(grid), "k--", lw=1.4,
        label="true $\\pm2$ sd (sd = 0.5 x)")
ax.plot(grid, true_mean(grid) - 2 * true_sd(grid), "k--", lw=1.4)
ax.plot(x[order], mu_b + 2 * sd_b, color="#1f77b4", lw=1.2, ls=":",
        label="baseline $\\pm2$ sd (constant)")
ax.plot(x[order], mu_b - 2 * sd_b, color="#1f77b4", lw=1.2, ls=":")
ax.set_xlabel("speed x [m/s]"); ax.set_ylabel("stopping distance y [m]")
ax.set_title(f"Selected by cross-validation: "
             f"{output_df['_source_selected'].iloc[0]}")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout(); fig.savefig("figures/selected.png", dpi=150)
print("\nredrew figures/selection.png and figures/selected.png from the records")

stamp: selected(d_mean=2,d_noise=1)
mean log predictive density on data/: baseline -4.7104   selected -4.3096
max |mu_selected - mu_model| = 0.00e+00 m
  (the selection landed on the same degrees the modeller guessed --
   the difference is that now there is a table saying why.)

redrew figures/selection.png and figures/selected.png from the records


---

## What the record remembers

In [8]:
print(f"rows: {len(data)}")
print("input columns :", list(input_df.columns))
print("output columns:", list(output_df.columns))

stamps = [c for c in output_df.columns if c.startswith("_source")]
if stamps:
    print("\nprovenance -- who wrote what:")
    for c in stamps:
        print(f"  {c:22s} {output_df[c].iloc[0]}")
else:
    print("\nno provenance columns yet")

print(f"\nstudy_selection/: {len(study)} candidates, "
      f"columns {list(study_out.columns)}")
print(f"  stamp: {study_out['_source_selection'].iloc[0]}")

# Everything on a slide has to come from one of these two records.  The
# held-out record data_test/ is deliberately NOT read here -- the reproduction
# gate is about data/, and blocks/holdout_check.py is the only thing that
# opens data_test/, once, at the very end.
print("\nfigures, and the record each is drawn from:")
for fig_name, src in [("baseline.png", "data/ (y_pred_baseline, sd_baseline)"),
                      ("model.png", "data/ (y_pred_model, sd_model)"),
                      ("selection.png", "study_selection/ (cv scores)"),
                      ("selected.png", "data/ (y_pred_selected, sd_selected)")]:
    print(f"  {fig_name:16s} <- {src}")

rows: 60
input columns : ['x']
output columns: ['y', 'y_pred_baseline', 'sd_baseline', '_source_baseline', 'y_pred_model', 'sd_model', '_source_modeler', 'y_pred_selected', 'sd_selected', '_source_selected']

provenance -- who wrote what:
  _source_baseline       baseline
  _source_modeler        modeler
  _source_selected       selected(d_mean=2,d_noise=1)

study_selection/: 12 candidates, columns ['cv_log_pred_density', 'cv_lpd_se', 'cv_rmse', 'n_params', 'n_fit_failures', '_source_selection']
  stamp: selection

figures, and the record each is drawn from:
  baseline.png     <- data/ (y_pred_baseline, sd_baseline)
  model.png        <- data/ (y_pred_model, sd_model)
  selection.png    <- study_selection/ (cv scores)
  selected.png     <- data/ (y_pred_selected, sd_selected)
